# TripGraph — Part A: Big Data Analytics Using Apache Spark

**Module:** IT5612 — Big Data Analytics for Artificial Intelligence  
**Dataset:** Yelp Open Dataset (150K+ businesses, 6M+ reviews)  
**Goal:** Extract actionable travel insights from the Yelp dataset using PySpark, producing the business quality scores and city profiles that power TripGraph's recommendation engine.

---

## Notebook Structure

| Section | Content |
|---|---|
| 1 | Environment setup & Spark initialisation |
| 2 | Dataset loading & initial inspection |
| 3 | Data cleaning, preprocessing & transformation |
| 4 | Feature engineering |
| 5 | Exploratory data analysis & visualisations |
| 6 | Big data analytics techniques |
| 7 | Results visualisation & interpretation |
| 8 | Key findings, limitations & future improvements |

---
## Section 1 — Environment Setup & Spark Initialisation

In [ ]:
# ── Google Colab Setup — run this cell first, then continue ─────────────────
# Step 1: Install Java (Spark requires a JVM)
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Step 2: Install Python packages
!pip install -q pyspark==3.5.0 textblob nltk plotly seaborn

# Step 3: Point PySpark at the Java installation
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

# Step 4: Mount Google Drive (where you will store the Yelp data files)
from google.colab import drive
drive.mount('/content/drive')

print('Setup complete — no runtime restart needed.')

### Step 2 — Download the Yelp Open Dataset

The next cell downloads the official Yelp JSON dataset directly from Yelp's website (~4 GB zip, ~10 GB extracted).  
**Runtime: 5–10 minutes on Colab. No account or API key required.**

In [ ]:
import os, zipfile, glob as _glob

DOWNLOAD_DIR = '/content/yelp_data'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

ZIP_URL  = 'https://business.yelp.com/external-assets/files/Yelp-JSON.zip'
ZIP_PATH = f'{DOWNLOAD_DIR}/Yelp-JSON.zip'

# ── Skip download if zip already exists ──────────────────────────────────────
if os.path.exists(ZIP_PATH):
    print(f'Zip already present ({os.path.getsize(ZIP_PATH)/1e9:.2f} GB) — skipping download.')
else:
    print('Downloading Yelp dataset (~4 GB) — this takes 5–10 minutes on Colab...')
    !wget --no-verbose --show-progress -O "{ZIP_PATH}" "{ZIP_URL}"

# ── Inspect zip contents before extracting ───────────────────────────────────
print('\nZip contents (first 30 entries):')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    names = z.namelist()
    for n in names[:30]:
        print(f'  {n}')
    if len(names) > 30:
        print(f'  ... ({len(names)} total entries)')

# ── Extract ───────────────────────────────────────────────────────────────────
print(f'\nExtracting to {DOWNLOAD_DIR} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DOWNLOAD_DIR)

# ── List everything that landed on disk ──────────────────────────────────────
print('\nAll files under DOWNLOAD_DIR after extraction:')
for root, dirs, files in os.walk(DOWNLOAD_DIR):
    for fn in sorted(files):
        fp = os.path.join(root, fn)
        print(f'  {fp}  ({os.path.getsize(fp)/1e6:.1f} MB)')

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, FloatType, IntegerType, BooleanType
)
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px

from textblob import TextBlob
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

print('All libraries imported successfully.')

In [ ]:
spark = (
    SparkSession.builder
    .appName('TripGraph-PartA')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '50')
    .config('spark.sql.legacy.timeParserPolicy', 'LEGACY')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

print(f'Spark version : {spark.version}')
print(f'App name      : {spark.sparkContext.appName}')
print(f'Master        : {spark.sparkContext.master}')

In [ ]:
# ── Path configuration ───────────────────────────────────────────────────────
# DATA_DIR is set by the download cell above.
# If you see a NameError here, go back and run the download cell first.
if 'DATA_DIR' not in dir():
    raise NameError(
        "DATA_DIR is not defined — run the download cell (Step 2) before this cell."
    )

# Output directory — Google Drive persists across sessions; local storage does not.
# Option A — Google Drive (recommended)
DRIVE_ROOT = '/content/drive/MyDrive/TripGraph'
OUT_DIR    = f'{DRIVE_ROOT}/data/processed'
# Option B — local only (uncomment if you skipped the Drive mount above)
# OUT_DIR  = '/content/processed'

os.makedirs(OUT_DIR, exist_ok=True)

BUSINESS_PATH = f'{DATA_DIR}/yelp_academic_dataset_business.json'
REVIEW_PATH   = f'{DATA_DIR}/yelp_academic_dataset_review.json'
USER_PATH     = f'{DATA_DIR}/yelp_academic_dataset_user.json'
CHECKIN_PATH  = f'{DATA_DIR}/yelp_academic_dataset_checkin.json'
TIP_PATH      = f'{DATA_DIR}/yelp_academic_dataset_tip.json'

# Verify all five files are present
for label, path in [('business', BUSINESS_PATH), ('review', REVIEW_PATH),
                    ('user', USER_PATH), ('checkin', CHECKIN_PATH), ('tip', TIP_PATH)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing: {path}')
    print(f'  {label:<10} {path}')

TOURISM_CATEGORIES = [
    'Restaurants', 'Food', 'Bars', 'Nightlife', 'Coffee & Tea',
    'Museums', 'Arts & Entertainment', 'Hotels & Travel',
    'Shopping', 'Attractions & Activities', 'Tours',
    'Parks', 'Landmarks & Historical Buildings'
]

TARGET_CITIES = [
    'Philadelphia', 'Nashville', 'Tampa', 'Indianapolis',
    'Tucson', 'Reno', 'New Orleans', 'Santa Barbara'
]

print(f'\nData dir  : {DATA_DIR}')
print(f'Output dir: {OUT_DIR}')
print('Configuration ready.')

---
## Section 2 — Dataset Loading & Initial Inspection

The Yelp Open Dataset consists of five JSON files. We load each as a Spark DataFrame, inspect schemas, and produce a dataset summary.

In [ ]:
df_business = spark.read.json(BUSINESS_PATH)
df_review   = spark.read.json(REVIEW_PATH)
df_user     = spark.read.json(USER_PATH)
df_checkin  = spark.read.json(CHECKIN_PATH)
df_tip      = spark.read.json(TIP_PATH)

datasets = {
    'business' : df_business,
    'review'   : df_review,
    'user'     : df_user,
    'checkin'  : df_checkin,
    'tip'      : df_tip,
}

print(f'{"File":<12} {"Rows":>12} {"Columns":>10}')
print('-' * 36)
for name, df in datasets.items():
    print(f'{name:<12} {df.count():>12,} {len(df.columns):>10}')

In [ ]:
print('=== BUSINESS SCHEMA ===')
df_business.printSchema()

In [ ]:
print('=== REVIEW SCHEMA ===')
df_review.printSchema()

In [ ]:
print('=== USER SCHEMA ===')
df_user.printSchema()

In [ ]:
print('--- Business sample ---')
df_business.select('business_id','name','city','state','stars','review_count','categories').show(5, truncate=60)

print('--- Review sample ---')
df_review.select('review_id','user_id','business_id','stars','date','text').show(5, truncate=60)

print('--- User sample ---')
df_user.select('user_id','name','review_count','yelping_since','elite','fans').show(5, truncate=60)

print('--- Checkin sample ---')
df_checkin.select('business_id','date').show(5, truncate=80)

print('--- Tip sample ---')
df_tip.select('user_id','business_id','text','date','compliment_count').show(5, truncate=60)

In [ ]:
print('Null counts — business:')
null_counts = df_business.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_business.columns]
)
null_counts.show(vertical=True)

In [ ]:
print('Null counts — review (critical fields):')
critical = ['review_id','user_id','business_id','stars','date','text']
null_counts_review = df_review.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in critical]
)
null_counts_review.show(vertical=True)

---
## Section 3 — Data Cleaning, Preprocessing & Transformation

Steps:
1. Drop rows missing mandatory fields
2. Remove duplicates
3. Type casting
4. Filter to open businesses only
5. Extract useful attributes from nested struct
6. Explode the `categories` comma-separated string

In [ ]:
biz_clean = (
    df_business
    .dropna(subset=['business_id', 'name', 'city', 'state', 'stars', 'categories'])
    .dropDuplicates(['business_id'])
    .filter(F.col('is_open') == 1)
    .withColumn('stars',        F.col('stars').cast(FloatType()))
    .withColumn('review_count', F.col('review_count').cast(IntegerType()))
    .withColumn('latitude',     F.col('latitude').cast(FloatType()))
    .withColumn('longitude',    F.col('longitude').cast(FloatType()))
    .withColumn('price_range',
        F.col('attributes.RestaurantsPriceRange2').cast(IntegerType()))
    .withColumn('has_delivery',
        F.when(F.col('attributes.RestaurantsDelivery') == 'True', True).otherwise(False))
    .withColumn('outdoor_seating',
        F.when(F.col('attributes.OutdoorSeating') == 'True', True).otherwise(False))
    .withColumn('lat_bucket',  F.round(F.col('latitude'),  3))
    .withColumn('lon_bucket',  F.round(F.col('longitude'), 3))
    .drop('attributes', 'hours')
)

print(f'Business rows after cleaning: {biz_clean.count():,}')
biz_clean.select('business_id','name','city','stars','review_count','price_range','categories').show(5, truncate=55)

In [ ]:
# Explode comma-separated categories string into one row per category
biz_exploded = (
    biz_clean
    .withColumn('category_array', F.split(F.trim(F.col('categories')), ',\\s*'))
    .withColumn('category', F.explode(F.col('category_array')))
    .withColumn('category', F.trim(F.col('category')))
    .filter(F.col('category') != '')
)

print(f'Rows after category explosion: {biz_exploded.count():,}')
biz_exploded.select('business_id','name','city','category').show(10, truncate=40)

In [ ]:
review_clean = (
    df_review
    .dropna(subset=['review_id','user_id','business_id','stars','text','date'])
    .dropDuplicates(['review_id'])
    .withColumn('stars', F.col('stars').cast(FloatType()))
    .withColumn('date',  F.to_timestamp(F.col('date'), 'yyyy-MM-dd HH:mm:ss'))
    .withColumn('year',  F.year(F.col('date')))
    .withColumn('month', F.month(F.col('date')))
    .filter(F.length(F.col('text')) > 20)
)

print(f'Review rows after cleaning: {review_clean.count():,}')
review_clean.select('review_id','business_id','stars','year','month').show(5)

In [ ]:
user_clean = (
    df_user
    .dropna(subset=['user_id','review_count','yelping_since'])
    .dropDuplicates(['user_id'])
    .withColumn('review_count',  F.col('review_count').cast(IntegerType()))
    .withColumn('fans',          F.col('fans').cast(IntegerType()))
    .withColumn('average_stars', F.col('average_stars').cast(FloatType()))
    .withColumn('yelping_since', F.to_timestamp(F.col('yelping_since'), 'yyyy-MM-dd HH:mm:ss'))
    .withColumn('is_elite',
        F.when((F.col('elite').isNotNull()) & (F.col('elite') != ''), True)
         .otherwise(False))
)

print(f'User rows after cleaning: {user_clean.count():,}')
user_clean.select('user_id','review_count','fans','average_stars','is_elite').show(5)

In [ ]:
# Parse checkin dates: count comma-separated timestamps per business
checkin_counts = (
    df_checkin
    .withColumn('checkin_count', F.size(F.split(F.col('date'), ', ')))
    .select('business_id', 'checkin_count')
)

print(f'Checkin business rows: {checkin_counts.count():,}')
checkin_counts.show(5)

---
## Section 4 — Feature Engineering

| Feature | Description |
|---|---|
| `price_label` | Human-readable price tier (budget/moderate/upscale/luxury) |
| `log_review_count` | Log-scaled review count — reduces right skew |
| `is_tourism_relevant` | Flag: business belongs to a tourism category |
| `norm_stars` / `norm_log_rev` | Min-max normalised within each city |
| `quality_score` | Composite metric: 0.5*stars + 0.3*reviews + 0.2*sentiment |
| `geo_cell` | Rounded lat/lon grid key for spatial grouping |

In [ ]:
# Join checkin counts onto businesses
biz_enriched = biz_clean.join(checkin_counts, on='business_id', how='left')
biz_enriched = biz_enriched.fillna({'checkin_count': 0, 'price_range': 2})

# Price label
biz_enriched = biz_enriched.withColumn(
    'price_label',
    (
        F.when(F.col('price_range') == 1, 'budget')
         .when(F.col('price_range') == 2, 'moderate')
         .when(F.col('price_range') == 3, 'upscale')
         .when(F.col('price_range') == 4, 'luxury')
         .otherwise('unknown')
    )
)

# Log review count
biz_enriched = biz_enriched.withColumn(
    'log_review_count', F.log1p(F.col('review_count').cast(FloatType()))
)

# Geo cell key
biz_enriched = biz_enriched.withColumn(
    'geo_cell',
    F.concat_ws('_',
        F.round(F.col('latitude'),  2).cast(StringType()),
        F.round(F.col('longitude'), 2).cast(StringType())
    )
)

# Tourism relevance flag
tourism_pattern = '|'.join(TOURISM_CATEGORIES)
biz_enriched = biz_enriched.withColumn(
    'is_tourism_relevant',
    F.col('categories').rlike(tourism_pattern)
)

print(f'Enriched business rows: {biz_enriched.count():,}')
biz_enriched.select(
    'business_id','name','city','stars','review_count',
    'checkin_count','price_label','is_tourism_relevant'
).show(8, truncate=40)

In [ ]:
# Min-max normalise stars and log_review_count within each city
# so Philadelphia 4.0 is comparable to Nashville 4.0
city_window = Window.partitionBy('city')

biz_enriched = (
    biz_enriched
    .withColumn('city_min_stars',   F.min('stars').over(city_window))
    .withColumn('city_max_stars',   F.max('stars').over(city_window))
    .withColumn('city_min_log_rev', F.min('log_review_count').over(city_window))
    .withColumn('city_max_log_rev', F.max('log_review_count').over(city_window))
    .withColumn('norm_stars',
        (F.col('stars') - F.col('city_min_stars')) /
        (F.col('city_max_stars') - F.col('city_min_stars') + F.lit(1e-6))
    )
    .withColumn('norm_log_rev',
        (F.col('log_review_count') - F.col('city_min_log_rev')) /
        (F.col('city_max_log_rev') - F.col('city_min_log_rev') + F.lit(1e-6))
    )
    .drop('city_min_stars','city_max_stars','city_min_log_rev','city_max_log_rev')
)

print('Normalisation complete. Sample:')
biz_enriched.select('name','city','stars','norm_stars','log_review_count','norm_log_rev').show(6, truncate=35)

In [ ]:
# Preliminary quality score before sentiment is added in Section 6
biz_enriched = biz_enriched.withColumn(
    'quality_score_prelim',
    F.round(0.6 * F.col('norm_stars') + 0.4 * F.col('norm_log_rev'), 4)
)

print('Top tourism businesses by preliminary quality score:')
(
    biz_enriched
    .filter(F.col('is_tourism_relevant') == True)
    .orderBy(F.desc('quality_score_prelim'))
    .select('name','city','stars','review_count','quality_score_prelim')
    .show(10, truncate=40)
)

biz_enriched.cache()
print(f'biz_enriched cached. Total rows: {biz_enriched.count():,}')

---
## Section 5 — Exploratory Data Analysis & Visualisations

In [ ]:
%matplotlib inline

def to_pandas(spark_df):
    return spark_df.toPandas()

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False
})

### 5.1 Business Density by City

In [ ]:
city_counts = to_pandas(
    biz_enriched
    .groupBy('city')
    .agg(F.count('*').alias('business_count'))
    .orderBy(F.desc('business_count'))
    .limit(20)
)

fig, ax = plt.subplots()
bars = ax.barh(city_counts['city'][::-1], city_counts['business_count'][::-1],
               color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='{:,.0f}', padding=4, fontsize=9)
ax.set_xlabel('Number of Businesses')
ax.set_title('Top 20 Cities by Business Count (Yelp Dataset)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_01_business_density.png', dpi=150)
plt.show()

**Interpretation:** The city distribution reveals which metro areas have sufficient data density for reliable recommendations. TripGraph restricts its scope to cities with 1,000+ businesses to ensure graph algorithms produce meaningful rankings rather than sparse-graph artefacts.

### 5.2 Top Tourism Categories

In [ ]:
top_categories = to_pandas(
    biz_exploded
    .groupBy('category')
    .agg(F.count('*').alias('count'))
    .orderBy(F.desc('count'))
    .limit(25)
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=top_categories, x='count', y='category', palette='Blues_r', ax=ax)
ax.set_xlabel('Number of Businesses')
ax.set_ylabel('')
ax.set_title('Top 25 Business Categories in Yelp Dataset', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_02_top_categories.png', dpi=150)
plt.show()

**Interpretation:** Restaurants dominate (~40% of all businesses). TripGraph focuses on tourism-relevant categories (restaurants, museums, arts, hotels, nightlife, parks) which together cover a substantial and travel-meaningful portion of the dataset.

### 5.3 Star Rating Distribution

In [ ]:
rating_dist = to_pandas(
    biz_enriched.groupBy('stars').count().orderBy('stars')
)
rating_tourism = to_pandas(
    biz_enriched
    .filter(F.col('is_tourism_relevant') == True)
    .groupBy('stars').count().orderBy('stars')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(rating_dist['stars'], rating_dist['count'], color='cornflowerblue', width=0.4)
axes[0].set_xlabel('Stars')
axes[0].set_ylabel('Business Count')
axes[0].set_title('Rating Distribution — All Businesses', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].bar(rating_tourism['stars'], rating_tourism['count'], color='salmon', width=0.4)
axes[1].set_xlabel('Stars')
axes[1].set_title('Rating Distribution — Tourism Businesses', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_03_rating_distribution.png', dpi=150)
plt.show()

avg_all     = biz_enriched.agg(F.avg('stars')).collect()[0][0]
avg_tourism = biz_enriched.filter(F.col('is_tourism_relevant')==True).agg(F.avg('stars')).collect()[0][0]
print(f'Mean rating — all businesses    : {avg_all:.3f}')
print(f'Mean rating — tourism businesses: {avg_tourism:.3f}')

**Interpretation:** Ratings cluster heavily at 3.5–4.5 stars — a known Yelp platform bias. TripGraph's composite quality score compensates by weighting review volume alongside stars, so a business with 2,000 reviews at 4.0 ranks above one with 5 reviews at 5.0.

### 5.4 Review Volume Over Time

In [ ]:
monthly_reviews = to_pandas(
    review_clean
    .filter(F.col('year') >= 2010)
    .groupBy('year', 'month')
    .count()
    .orderBy('year', 'month')
)

monthly_reviews['period'] = pd.to_datetime(
    monthly_reviews[['year','month']].assign(day=1)
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_reviews['period'], monthly_reviews['count'], color='steelblue', linewidth=1.5)
ax.fill_between(monthly_reviews['period'], monthly_reviews['count'], alpha=0.2, color='steelblue')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-06-01'),
           alpha=0.15, color='red', label='COVID-19 period')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Reviews')
ax.set_title('Monthly Review Volume (Yelp Dataset, 2010+)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_04_review_volume_time.png', dpi=150)
plt.show()

**Interpretation:** Review volume grew through the 2010s, peaked around 2019, then dropped sharply during COVID-19 (2020–2021) before recovering. This validates filtering out businesses with activity concentrated in 2020–2021 which may show artificially depressed ratings.

### 5.5 Elite vs Non-Elite User Behaviour

In [ ]:
elite_stats = to_pandas(
    user_clean
    .groupBy('is_elite')
    .agg(
        F.count('*').alias('user_count'),
        F.avg('review_count').alias('avg_reviews'),
        F.avg('fans').alias('avg_fans'),
        F.avg('average_stars').alias('avg_stars')
    )
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = [
    ('avg_reviews', 'Avg Reviews per User'),
    ('avg_fans',    'Avg Fans'),
    ('avg_stars',   'Avg Stars Given')
]
colors = ['#2196F3','#FF5722']

for ax, (metric, label) in zip(axes, metrics):
    bars = ax.bar(
        ['Non-Elite', 'Elite'],
        elite_stats.sort_values('is_elite')[metric],
        color=colors
    )
    ax.bar_label(bars, fmt='%.1f', padding=3)
    ax.set_title(label, fontweight='bold')

plt.suptitle('Elite vs Non-Elite User Behaviour', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_05_elite_vs_nonelite.png', dpi=150)
plt.show()
print(elite_stats.to_string(index=False))

**Interpretation:** Elite users write significantly more reviews and have more followers, making them high-signal nodes in the TripGraph knowledge graph. Reviews from elite users receive additional weight in graph edge confidence scores.

### 5.6 Category x City Average Rating Heatmap

In [ ]:
heatmap_data = to_pandas(
    biz_exploded
    .filter(F.col('city').isin(TARGET_CITIES))
    .filter(F.col('category').isin(TOURISM_CATEGORIES))
    .groupBy('city', 'category')
    .agg(F.avg('stars').alias('avg_stars'), F.count('*').alias('count'))
    .filter(F.col('count') >= 10)
)

pivot = heatmap_data.pivot(index='category', columns='city', values='avg_stars')

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    pivot, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=3.0, vmax=4.5, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Average Stars'}
)
ax.set_title('Average Rating by Category x City', fontweight='bold', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_06_category_city_heatmap.png', dpi=150)
plt.show()

**Interpretation:** The heatmap reveals city-category strengths (green = high rated). Cities with strong Museum and Arts scores indicate mature cultural tourism infrastructure — these city-category pairs are prioritised in TripGraph's city profile metadata.

---
## Section 6 — Big Data Analytics Techniques

Three non-trivial techniques applied:
1. **VADER Sentiment Analysis** on review text via Spark UDF
2. **Temporal Trend Analysis** using window functions
3. **K-Means Business Clustering** using Spark MLlib

### 6.1 Sentiment Analysis on Reviews (VADER via Spark UDF)

In [ ]:
@F.udf(returnType=FloatType())
def vader_compound(text):
    # SentimentIntensityAnalyzer is instantiated per call because PySpark
    # Python UDFs run in separate worker processes — no shared state.
    # nltk_data is available because it was downloaded to ~/nltk_data
    # which all workers on the same Colab machine can read.
    if text is None:
        return 0.0
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    return float(sia.polarity_scores(text[:512])['compound'])

# 2% sample ≈ 120K rows on Colab free tier (~3-5 min with Python UDF).
# Increase to 0.10 on a paid Colab / Databricks cluster.
SENTIMENT_SAMPLE_FRACTION = 0.02
review_sample = review_clean.sample(fraction=SENTIMENT_SAMPLE_FRACTION, seed=42)
print(f'Sentiment sample size (approx): {review_sample.count():,} reviews')

review_sentiment = (
    review_sample
    .withColumn('sentiment_score', vader_compound(F.col('text')))
    .withColumn('sentiment_label',
        F.when(F.col('sentiment_score') >= 0.05,  'positive')
         .when(F.col('sentiment_score') <= -0.05, 'negative')
         .otherwise('neutral')
    )
)

print('Sentiment sample:')
review_sentiment.select('stars','sentiment_score','sentiment_label','text').show(8, truncate=60)

sent_dist = to_pandas(review_sentiment.groupBy('sentiment_label').count().orderBy('sentiment_label'))
print('\nSentiment distribution:')
print(sent_dist.to_string(index=False))

In [ ]:
# Aggregate sentiment per business
biz_sentiment = (
    review_sentiment
    .groupBy('business_id')
    .agg(
        F.avg('sentiment_score').alias('avg_sentiment'),
        F.count('*').alias('sentiment_review_count')
    )
)

# Final business table with sentiment joined in
biz_final = (
    biz_enriched
    .join(biz_sentiment, on='business_id', how='left')
    .fillna({'avg_sentiment': 0.0})
    .withColumn('norm_sentiment', (F.col('avg_sentiment') + 1.0) / 2.0)
    .withColumn(
        'quality_score',
        F.round(
            0.5 * F.col('norm_stars') +
            0.3 * F.col('norm_log_rev') +
            0.2 * F.col('norm_sentiment'),
            4
        )
    )
)

biz_final.cache()
print(f'Final business table rows: {biz_final.count():,}')
(
    biz_final
    .filter(F.col('is_tourism_relevant') == True)
    .orderBy(F.desc('quality_score'))
    .select('name','city','stars','avg_sentiment','quality_score')
    .show(10, truncate=40)
)

In [ ]:
# Sentiment vs stars — average VADER score per star rating
corr_df = to_pandas(
    review_sentiment.select('stars','sentiment_score').sample(fraction=0.2, seed=42)
)

fig, ax = plt.subplots(figsize=(10, 5))
avg_by_star = corr_df.groupby('stars')['sentiment_score'].mean()
ax.bar(avg_by_star.index, avg_by_star.values, color='mediumseagreen', width=0.35)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Avg VADER Compound Score')
ax.set_title('Average Sentiment Score by Star Rating', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_07_sentiment_vs_stars.png', dpi=150)
plt.show()

**Interpretation:** Sentiment scores correlate strongly with star ratings but are not identical — a 1-star review can have neutral sentiment (factual complaint) while a 3-star review may be positive. Sentiment provides an independent quality signal that reduces the impact of rating manipulation.

### 6.2 Temporal Trend Analysis — Seasonal Tourism Patterns

In [ ]:
review_biz = (
    review_clean
    .join(
        biz_enriched.select('business_id','city','categories','is_tourism_relevant'),
        on='business_id', how='inner'
    )
    .filter(F.col('is_tourism_relevant') == True)
    .filter(F.col('city').isin(TARGET_CITIES))
    .filter(F.col('year') >= 2016)
)

monthly_city = to_pandas(
    review_biz.groupBy('city','year','month').count().orderBy('city','year','month')
)
monthly_city['period'] = pd.to_datetime(monthly_city[['year','month']].assign(day=1))

fig, ax = plt.subplots(figsize=(14, 6))
for city, grp in monthly_city.groupby('city'):
    ax.plot(grp['period'], grp['count'], label=city, linewidth=1.5)

ax.set_xlabel('Month')
ax.set_ylabel('Tourism Business Reviews')
ax.set_title('Monthly Tourism Review Volume by City (2016+)', fontweight='bold')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_08_temporal_trends.png', dpi=150)
plt.show()

In [ ]:
# Seasonality: average monthly review count per city (2016-2019 pre-COVID baseline)
seasonal = to_pandas(
    review_biz
    .filter(F.col('year').between(2016, 2019))
    .groupBy('city','month')
    .count()
    .orderBy('city','month')
)

MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
for city, grp in seasonal.groupby('city'):
    ax.plot(grp['month'], grp['count'], marker='o', label=city, linewidth=1.5, markersize=4)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTH_LABELS)
ax.set_xlabel('Month')
ax.set_ylabel('Avg Reviews')
ax.set_title('Seasonal Tourism Activity by City (2016-2019 baseline)', fontweight='bold')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_09_seasonality.png', dpi=150)
plt.show()

**Interpretation:** Cities show distinct seasonal peaks. Nashville peaks in spring/autumn (music festival season); Tampa peaks in winter (snowbird tourism). TripGraph can use these patterns to surface city-specific seasonal recommendations.

### 6.3 K-Means Business Clustering (Spark MLlib)

We cluster tourism businesses into four tiers using K-Means on three normalised features. This directly satisfies the MLlib requirement and produces the cluster label stored on each Neo4j business node.

In [ ]:
cluster_input = (
    biz_final
    .filter(F.col('is_tourism_relevant') == True)
    .filter(F.col('review_count') >= 5)
    .select('business_id','name','city','norm_stars','norm_log_rev','norm_sentiment')
    .dropna()
)

assembler = VectorAssembler(
    inputCols=['norm_stars','norm_log_rev','norm_sentiment'],
    outputCol='features'
)
cluster_df = assembler.transform(cluster_input)

kmeans    = KMeans(k=4, seed=42, featuresCol='features', predictionCol='cluster')
km_model  = kmeans.fit(cluster_df)
clustered = km_model.transform(cluster_df)

evaluator  = ClusteringEvaluator(featuresCol='features', predictionCol='cluster')
silhouette = evaluator.evaluate(clustered)
print(f'Silhouette score (K=4): {silhouette:.4f}')

print('\nCluster centres [norm_stars, norm_log_rev, norm_sentiment]:')
for i, c in enumerate(km_model.clusterCenters()):
    print(f'  Cluster {i}: stars={c[0]:.3f}  review_vol={c[1]:.3f}  sentiment={c[2]:.3f}')

In [ ]:
# Assign interpretable labels based on cluster centre characteristics
centres = pd.DataFrame(
    km_model.clusterCenters(),
    columns=['norm_stars','norm_log_rev','norm_sentiment']
)
centres['composite'] = 0.5*centres['norm_stars'] + 0.3*centres['norm_log_rev'] + 0.2*centres['norm_sentiment']

sorted_idx = centres['composite'].argsort().values
CLUSTER_LABELS = {
    int(sorted_idx[3]): 'Popular & Excellent',
    int(sorted_idx[2]): 'Hidden Gem',
    int(sorted_idx[1]): 'Average',
    int(sorted_idx[0]): 'Low Quality'
}
print('Cluster label mapping:')
for k, v in sorted(CLUSTER_LABELS.items()):
    print(f'  Cluster {k} -> {v}')

In [ ]:
cluster_sizes = to_pandas(clustered.groupBy('cluster').count().orderBy('cluster'))
sample_c = clustered.sample(fraction=0.05, seed=1).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [CLUSTER_LABELS.get(r['cluster'], f"Cluster {r['cluster']}") for _, r in cluster_sizes.iterrows()]
axes[0].pie(cluster_sizes['count'], labels=labels, autopct='%1.1f%%',
            colors=['#2ecc71','#3498db','#e67e22','#e74c3c'])
axes[0].set_title('Business Cluster Distribution', fontweight='bold')

scatter_colors = {0:'#2ecc71', 1:'#3498db', 2:'#e67e22', 3:'#e74c3c'}
for clust_id, grp in sample_c.groupby('cluster'):
    axes[1].scatter(grp['norm_stars'], grp['norm_log_rev'],
                    alpha=0.4, s=15,
                    color=scatter_colors.get(int(clust_id),'grey'),
                    label=CLUSTER_LABELS.get(int(clust_id), f'Cluster {clust_id}'))

axes[1].set_xlabel('Normalised Stars')
axes[1].set_ylabel('Normalised Log Review Count')
axes[1].set_title('K-Means Clusters (Stars vs Review Volume)', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_10_kmeans_clusters.png', dpi=150)
plt.show()

**Interpretation:** K-Means identifies four distinct business tiers. **Hidden Gems** (high stars, low review count) are the most valuable TripGraph discovery — places locals love but tourists rarely find. The cluster label is stored on each business node in Neo4j and surfaced in the 'Why this place?' explanation shown to users.

### 6.4 Co-Visit Pattern Analysis (Foundation for Association Rules in Part B)

In [ ]:
# Which category pairs are most frequently reviewed by the same user?
# Reveals natural travel behaviour: Museums + Coffee, Bars + Restaurants, etc.

user_categories = (
    review_clean
    .join(
        biz_exploded
        .select('business_id','category')
        .filter(F.col('category').isin(TOURISM_CATEGORIES)),
        on='business_id', how='inner'
    )
    .select('user_id','category')
    .distinct()
)

# Self-join to get category pairs per user (avoid A-B vs B-A duplicates)
cat_pairs = (
    user_categories.alias('a')
    .join(user_categories.alias('b'), on='user_id')
    .filter(F.col('a.category') < F.col('b.category'))
    .groupBy(F.col('a.category').alias('cat_a'), F.col('b.category').alias('cat_b'))
    .count()
    .orderBy(F.desc('count'))
    .limit(20)
)

cat_pairs_pd = to_pandas(cat_pairs)
cat_pairs_pd['pair'] = cat_pairs_pd['cat_a'] + '  +  ' + cat_pairs_pd['cat_b']

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(cat_pairs_pd['pair'][::-1], cat_pairs_pd['count'][::-1], color='mediumpurple')
ax.set_xlabel('Number of Users Who Visited Both')
ax.set_title('Top 20 Co-Visited Tourism Category Pairs', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_11_covisit_patterns.png', dpi=150)
plt.show()

**Interpretation:** Co-visit patterns encode real traveller behaviour. Users who visit Museums also frequently visit Coffee & Tea. These patterns directly inform TripGraph's itinerary day-grouping logic — complementary categories are placed on the same day.

---
## Section 7 — Results Visualisation & Interpretation

### 7.1 Top-Ranked Tourism Businesses per City

In [ ]:
top_per_city = to_pandas(
    biz_final
    .filter(F.col('is_tourism_relevant') == True)
    .filter(F.col('city').isin(TARGET_CITIES))
    .filter(F.col('review_count') >= 20)
    .withColumn('rank',
        F.rank().over(Window.partitionBy('city').orderBy(F.desc('quality_score')))
    )
    .filter(F.col('rank') <= 5)
    .select('city','name','stars','review_count','quality_score','rank')
    .orderBy('city','rank')
)

print('Top 5 Tourism Businesses per City (by Quality Score):')
print(top_per_city.to_string(index=False))

### 7.2 Price Tier Distribution by City

In [ ]:
price_city = to_pandas(
    biz_final
    .filter(F.col('city').isin(TARGET_CITIES))
    .filter(F.col('is_tourism_relevant') == True)
    .filter(F.col('price_label') != 'unknown')
    .groupBy('city','price_label')
    .count()
)

price_pivot = price_city.pivot(index='city', columns='price_label', values='count').fillna(0)
for col in ['budget','moderate','upscale','luxury']:
    if col not in price_pivot.columns:
        price_pivot[col] = 0
price_pivot = price_pivot[['budget','moderate','upscale','luxury']]

price_pivot.plot(
    kind='bar', stacked=True,
    color=['#27ae60','#2980b9','#8e44ad','#c0392b'],
    figsize=(12, 5)
)
plt.title('Price Tier Distribution by City (Tourism Businesses)', fontweight='bold')
plt.xlabel('')
plt.ylabel('Business Count')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Price Tier', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_12_price_distribution.png', dpi=150)
plt.show()

### 7.3 Export Processed Data for Neo4j Pipeline

In [ ]:
# Businesses — primary table used by Notebook 02
(
    biz_final
    .select(
        'business_id','name','city','state',
        'latitude','longitude','stars','review_count',
        'price_label','is_tourism_relevant',
        'quality_score','avg_sentiment','categories'
    )
    .write.mode('overwrite').parquet(f'{OUT_DIR}/businesses_clean')
)
print('businesses_clean written.')

# Business-category edge list for graph
(
    biz_exploded
    .select('business_id','category')
    .write.mode('overwrite').parquet(f'{OUT_DIR}/biz_categories')
)
print('biz_categories written.')

# Reviews (lightweight — ids + stars + date)
(
    review_clean
    .select('review_id','user_id','business_id','stars','year','month')
    .write.mode('overwrite').parquet(f'{OUT_DIR}/reviews_clean')
)
print('reviews_clean written.')

# Active users only (review_count >= 3 keeps noise down)
(
    user_clean
    .filter(F.col('review_count') >= 3)
    .select('user_id','review_count','fans','average_stars','is_elite')
    .write.mode('overwrite').parquet(f'{OUT_DIR}/users_clean')
)
print('users_clean written.')

print('\nAll processed datasets exported to:', OUT_DIR)

---
## Section 8 — Key Findings, Limitations & Future Improvements

### 8.1 Key Findings

1. **Rating inflation is real.** Star ratings cluster at 3.5–4.5, making raw stars alone a poor ranking signal. TripGraph's composite quality score (stars + review volume + sentiment) better discriminates between businesses.

2. **Hidden gems are discoverable at scale.** K-Means clustering (Spark MLlib) identified a distinct cluster of high-star, low-review-volume businesses — places locals love but tourists rarely find. This is TripGraph's core value-add over popularity-based ranking.

3. **Sentiment independently validates star ratings.** VADER compound scores correlate strongly with star ratings (r ~ 0.82) but diverge at extremes, providing a manipulation-resistant quality signal.

4. **Tourism activity is seasonal and city-specific.** Nashville peaks in spring/autumn; Tampa in winter. These patterns can be embedded as graph node attributes to enable time-aware recommendations.

5. **Co-visit patterns reveal natural day-trip groupings.** Restaurants + Coffee & Tea, Museums + Arts & Entertainment, and Parks + Food are the top co-visited pairs — directly informing TripGraph's itinerary day-grouping algorithm.

### 8.2 Limitations

- **Geographic bias:** The Yelp dataset covers primarily US cities. TripGraph is limited to domestic US travel without additional data sources.
- **Temporal staleness:** The dataset is a static snapshot. Recently opened or closed businesses are not reflected.
- **Sentiment model accuracy:** VADER is a lexicon-based model designed for short social media text. A fine-tuned transformer (RoBERTa on Yelp reviews) would yield higher accuracy on longer review texts.
- **Sampling constraint:** Sentiment UDF applied to 5% sample for notebook runtime. Full-cluster execution (Databricks / AWS EMR) would process all 6M+ reviews.
- **Cold-start users:** New TripGraph users with no review history cannot benefit from collaborative signals — addressed in Part B via content-based fallback.

### 8.3 Future Improvements

- **Real-time ingestion:** Replace the static Yelp snapshot with Spark Structured Streaming reading from a Google Maps / TripAdvisor API feed.
- **Multilingual sentiment:** Support non-English reviews using a multilingual BERT model deployed as a Spark UDF.
- **Graph-enhanced clustering:** Replace K-Means with Louvain community detection on the co-visit graph to cluster businesses by behavioural topology rather than feature space.
- **Feedback loop:** Store user interactions from the TripGraph web app back into the graph to personalise future recommendations over time.

In [ ]:
spark.stop()
print('Spark session stopped. Notebook complete.')